## Width vs Depth [Part 1].ipynb

In [ ]:
import numpy as np
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
import torch.optim as optim
import cv2
from matplotlib.gridspec import GridSpec
from tqdm import tqdm
import os

import pickle
import copy
import os

In [ ]:
def denorm(p, size=960):
    result = np.zeros_like(p)
    result[..., 0] = (p[..., 0] + 1) * (size / 2)
    result[..., 1] = size - ((p[..., 1] + 1) * (size / 2))
    return result

def get_relu_joint(weight_1, weight_2, bias, extent=1):
    if np.abs(weight_2) < 1e-8: 
        x_intercept = -bias / weight_1
        return [[x_intercept, -extent], [x_intercept, extent]] if -extent <= x_intercept <= extent else []
    elif np.abs(weight_1) < 1e-8:
        y_intercept = -bias / weight_2
        return [[-extent, y_intercept], [extent, y_intercept]] if -extent <= y_intercept <= extent else []
    else:
        points = []
        for x in [-extent, extent]:
            y = (-x * weight_1 - bias) / weight_2
            if -extent <= y <= extent: points.append([x, y])
        for y in [-extent, extent]:
            x = (-y * weight_2 - bias) / weight_1
            if -extent <= x <= extent: points.append([x, y])
        unique_points = []
        for p in points:
            is_duplicate = False
            for existing in unique_points:
                if abs(p[0] - existing[0]) < 1e-8 and abs(p[1] - existing[1]) < 1e-8:
                    is_duplicate = True
                    break
            if not is_duplicate:
                unique_points.append(p)
        return unique_points

def viz_map_with_predictions(ax, map, X, y):
    probe=np.zeros((100,100,2))
    for j, xx in enumerate(np.linspace(-1, 1, 100)):
        for k, yy in enumerate(np.linspace(-1, 1, 100)):
            probe[j, k]=[yy,xx]
    probe=probe.reshape(100**2, -1)
    probe_logits=model(torch.tensor(probe).float())
    probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
    probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))
    
    ax.imshow(map.mean(2), cmap='gray')
    ax.imshow(np.flipud(probe_softmax[:,1].reshape(100, 100)), 
               extent=[0, 960, 960, 0],  # This maps to image coordinates
               alpha=0.7,
               cmap='viridis')
    
    X_denorm=denorm(X[:1000,:])
    # plt.scatter(X_denorm[:,0], X_denorm[:,1], c=y[:1000], s=2, alpha=0.8)
    labels=y[:1000]
    y_hat=torch.argmax(model(torch.tensor(X[:1000,:]).float()).detach(),1).numpy()
    tp_mask=np.logical_and(labels==1, y_hat==1)
    fp_mask=np.logical_and(labels==0, y_hat==1)
    fn_mask=np.logical_and(labels==1, y_hat==0)
    tn_mask=np.logical_and(labels==0, y_hat==0)
    ax.scatter(X_denorm[tp_mask,0], X_denorm[tp_mask,1], c='g', s=3, alpha=0.8)
    ax.scatter(X_denorm[fp_mask,0], X_denorm[fp_mask,1], c='r', marker='x', s=5, alpha=0.8)
    ax.scatter(X_denorm[fn_mask,0], X_denorm[fn_mask,1], c='orange', marker='x', s=5, alpha=0.8)
    ax.scatter(X_denorm[tn_mask,0], X_denorm[tn_mask,1], c='purple', marker='o', s=5, alpha=0.8)

def save_viz(model, save_path=None):
    plt.clf()
    probe=np.zeros((100,100,2))
    for j, xx in enumerate(np.linspace(-1, 1, 100)):
        for k, yy in enumerate(np.linspace(-1, 1, 100)):
            probe[j, k]=[yy,xx]
    probe=probe.reshape(100**2, -1)
    
    fig=plt.figure(0,(10,10)); 
    ax=fig.add_subplot(111)
    probe_logits=model(torch.tensor(probe).float())
    probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
    probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))
    
    ax.imshow(map.mean(2), cmap='gray')
    ax.imshow(np.flipud(np.argmin(probe_logits, 2)), 
               extent=[0, 960, 960, 0],  # This maps to image coordinates
               alpha=0.7,
               cmap='viridis')
    # ax.imshow(np.flipud(probe_softmax[:,1].reshape(100, 100)), 
    #            extent=[0, 960, 960, 0],  # This maps to image coordinates
    #            alpha=0.7,
    #            cmap='viridis')
    if save_path is not None: plt.savefig(save_path)

def extract_weights_to_numpy(model, layer_indices=None, output_file=None):
    """
    Extract PyTorch model weights and biases to NumPy arrays with copy-paste ready code.
    
    Args:
        model: PyTorch model
        layer_indices: List of layer indices to extract (if None, extracts all layers with weights)
        output_file: Optional file path to save the output code
    
    Returns:
        String containing the NumPy array definitions ready to copy-paste
    """
    
    lines = []
    lines.append("import numpy as np\n")
    
    # Auto-detect layers if not specified
    if layer_indices is None:
        layer_indices = []
        for i, layer in enumerate(model.model):
            if hasattr(layer, 'weight'):
                layer_indices.append(i)
    
    # Extract weights and biases
    weight_counter = 1
    for idx in layer_indices:
        layer = model.model[idx]
        
        if hasattr(layer, 'weight'):
            # Extract weight
            weight_np = layer.weight.detach().numpy()
            weight_repr = np.array2string(weight_np, 
                                        separator=', ', 
                                        formatter={'float_kind': lambda x: f"{x:g}"})
            lines.append(f"w{weight_counter} = np.array({weight_repr}, dtype=np.float32)")
            
            # Extract bias if it exists
            if hasattr(layer, 'bias') and layer.bias is not None:
                bias_np = layer.bias.detach().numpy()
                bias_repr = np.array2string(bias_np, 
                                          separator=', ', 
                                          formatter={'float_kind': lambda x: f"{x:g}"})
                lines.append(f"b{weight_counter} = np.array({bias_repr}, dtype=np.float32)")
            
            weight_counter += 1
    
    output = '\n'.join(lines)
    
    # Save to file if specified
    if output_file:
        with open(output_file, 'w') as f:
            f.write(output)
        print(f"Weights saved to {output_file}")
    
    # Print to console
    print(output)
    return output

In [ ]:
map=cv2.imread('Baarle-Nassau_-_Baarle-Hertog-en.svg.png')[:,:,(2,1,0)]

belgium_color=np.array([251,234,81])
netherlands_color=np.array([255, 255, 228])

netherlands_region=((map-netherlands_color)**2).sum(-1)<10
belgium_region=((map-belgium_color)**2).sum(-1)<10000

num_points_to_sample=10000 #10000

b_coords=np.array(np.where(belgium_region)).T.astype('float')
n_coords=np.array(np.where(netherlands_region)).T.astype('float')

#flip and normalize
belgium_coords_all=np.zeros_like(b_coords)
netherlands_coords_all=np.zeros_like(n_coords)

belgium_coords_all[:,0]=b_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
belgium_coords_all[:,1]=(960-b_coords[:,0])/(960/2)-1
netherlands_coords_all[:,0]=n_coords[:,1]/(960/2)-1 #Scale to -1 to 1 for now
netherlands_coords_all[:,1]=(960-n_coords[:,0])/(960/2)-1

## Hmm to balance to classes or do more area proportional? Not sure yet. 
belgium_coords=belgium_coords_all[np.random.choice(len(belgium_coords_all), num_points_to_sample), :]
netherlands_coords=netherlands_coords_all[np.random.choice(len(netherlands_coords_all), num_points_to_sample), :]

X=np.vstack((netherlands_coords, belgium_coords))
y=np.concatenate((np.zeros(len(netherlands_coords)), np.ones(len(belgium_coords)))).astype('int')

rI=np.arange(len(y))
np.random.shuffle(rI)
X=X[rI,:]
y=y[rI]

In [ ]:
class BaarleNet(nn.Module):
    def __init__(self, hidden_layers=[64]):
        super(BaarleNet, self).__init__()
        layers = [nn.Linear(2, hidden_layers[0]), nn.ReLU()]
        for i in range(len(hidden_layers)-1):
            layers.append(nn.Linear(hidden_layers[i], hidden_layers[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_layers[-1], 2))
        self.layers=layers
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

## [2]

In [ ]:
# accuracies=[]
# for random_seed in range(32):
#     torch.manual_seed(random_seed)
#     X_tensor = torch.FloatTensor(X)
#     y_tensor = torch.tensor(y)
    
#     # Initialize model and optimizer
#     model = BaarleNet([2])
    
#     #Larger weight init
#     # bound=2.5
#     # with torch.no_grad():
#     #     model.model[0].weight.uniform_(-bound, bound)
#     #     model.model[0].bias.uniform_(-bound, bound)  
#     #     model.model[2].weight.uniform_(-bound, bound)
#     #     model.model[2].bias.uniform_(-bound, bound)  
#     #     if len(model.model)>3: #ONLY SETUP FOR 2 hidden layers so far!
#     #         model.model[4].weight.uniform_(-bound, bound)
#     #         model.model[4].bias.uniform_(-bound, bound) 

#     criterion = nn.CrossEntropyLoss()
#     # optimizer = optim.SGD(model.parameters(), lr=0.25)
#     optimizer = optim.Adam(model.parameters(), lr=0.0025) #0.005
    
#     num_epochs = 12000
#     for epoch in range(num_epochs):
#         # Batch
#         # outputs = model(X_tensor)
#         # loss = criterion(outputs, y_tensor)
    
#         #Stochastic
#         outputs = model(torch.tensor(X[epoch%len(y)]).float())
#         loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

#         # Backward and optimize
#         optimizer.zero_grad()
#         loss.backward()

#         optimizer.step()
        
#     outputs_batch = model(X_tensor)
#     accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
#     print(f'Random Seed [{random_seed}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')
#     accuracies.append(accuracy)

In [ ]:
random_seed=25
torch.manual_seed(random_seed)  

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Initialize model and optimizer
model = BaarleNet([2])
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.25)
optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005
# optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005

num_epochs = 12000
for epoch in range(num_epochs):
    # Batch
    # outputs = model(X_tensor)
    # loss = criterion(outputs, y_tensor)

    #Stochastic
    outputs = model(torch.tensor(X[epoch%len(y)]).float())
    loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

    #Worth trying minibatch?
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # if epoch==307: break #Stop early here so weights don't get updated before we poke around
    
    optimizer.step()
    if (epoch+1) % (num_epochs//10) == 0:
        outputs_batch = model(X_tensor)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
fig=plt.figure(0,(8,8))
ax=fig.add_subplot(111)

probe=np.zeros((100,100,2))
for j, xx in enumerate(np.linspace(-1, 1, 100)):
    for k, yy in enumerate(np.linspace(-1, 1, 100)):
        probe[j, k]=[yy,xx]
probe=probe.reshape(100**2, -1)
probe_logits=model(torch.tensor(probe).float())
probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))

ax.imshow(map.mean(2), cmap='gray')
ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
           extent=[0, 960, 960, 0],  # This maps to image coordinates
           alpha=0.7,
           cmap='viridis')

In [ ]:
torch.save(model.state_dict(), '../models/2_1.pth')

In [ ]:
_=extract_weights_to_numpy(model)

## 2x2

In [ ]:
# accuracies=[]
# for random_seed in range(20):
#     torch.manual_seed(random_seed)
#     X_tensor = torch.FloatTensor(X)
#     y_tensor = torch.tensor(y)
    
#     # Initialize model and optimizer
#     model = BaarleNet([2,2])
    
#     #Larger weight init
#     bound=2.5
#     with torch.no_grad():
#         model.model[0].weight.uniform_(-bound, bound)
#         model.model[0].bias.uniform_(-bound, bound)  
#         model.model[2].weight.uniform_(-bound, bound)
#         model.model[2].bias.uniform_(-bound, bound)  
#         if len(model.model)>3: #ONLY SETUP FOR 2 hidden layers so far!
#             model.model[4].weight.uniform_(-bound, bound)
#             model.model[4].bias.uniform_(-bound, bound) 

#     criterion = nn.CrossEntropyLoss()
#     # optimizer = optim.SGD(model.parameters(), lr=0.25)
#     optimizer = optim.Adam(model.parameters(), lr=0.0025) #0.005
    
#     num_epochs = 12000
#     for epoch in range(num_epochs):
#         # Batch
#         # outputs = model(X_tensor)
#         # loss = criterion(outputs, y_tensor)
    
#         #Stochastic
#         outputs = model(torch.tensor(X[epoch%len(y)]).float())
#         loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

#         # Backward and optimize
#         optimizer.zero_grad()
#         loss.backward()

#         optimizer.step()
        
#     outputs_batch = model(X_tensor)
#     accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
#     print(f'Random Seed [{random_seed}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')
#     accuracies.append(accuracy)

- Ok wow 2x2 model seems kind a lottery ticket-y
- So the narrative angle then might be comparing like a 32 or 64 neuron shallow model to distributing those neurons?
- I think I can collapse down to somethign small to make points as I need to?
- Yeah so it takes a long time to find a lottery ticket, and the decision boundary is not that cool once you do.
- Yeah what's like the minimum interesting depth example?
- Like "oh my god, if you take these N neurons from a single layer and spread them accross J layers you can do whatever the fuck you want." 

In [ ]:
random_seed=315
torch.manual_seed(random_seed)  

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Initialize model and optimizer
model = BaarleNet([2,2])
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.25)
optimizer = optim.Adam(model.parameters(), lr=0.002) #0.005
# optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005

num_epochs = 60000
for epoch in range(num_epochs):
    # Batch
    # outputs = model(X_tensor)
    # loss = criterion(outputs, y_tensor)

    #Stochastic
    outputs = model(torch.tensor(X[epoch%len(y)]).float())
    loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

    #Worth trying minibatch?
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # if epoch==307: break #Stop early here so weights don't get updated before we poke around
    
    optimizer.step()
    if (epoch+1) % (num_epochs//10) == 0:
        outputs_batch = model(X_tensor)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
fig=plt.figure(0,(8,8))
ax=fig.add_subplot(111)

probe=np.zeros((100,100,2))
for j, xx in enumerate(np.linspace(-1, 1, 100)):
    for k, yy in enumerate(np.linspace(-1, 1, 100)):
        probe[j, k]=[yy,xx]
probe=probe.reshape(100**2, -1)
probe_logits=model(torch.tensor(probe).float())
probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))

ax.imshow(map.mean(2), cmap='gray')
ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
           extent=[0, 960, 960, 0],  # This maps to image coordinates
           alpha=0.7,
           cmap='viridis')

In [ ]:
_=extract_weights_to_numpy(model)

In [ ]:
# torch.save(model.state_dict(), '../models/2_2_1.pth')

## 3x3

In [ ]:
# accuracies=[]
# for random_seed in range(32):
#     torch.manual_seed(random_seed)
#     X_tensor = torch.FloatTensor(X)
#     y_tensor = torch.tensor(y)
    
#     # Initialize model and optimizer
#     model = BaarleNet([3,3])
    
#     #Larger weight init
#     # bound=2.5
#     # with torch.no_grad():
#     #     model.model[0].weight.uniform_(-bound, bound)
#     #     model.model[0].bias.uniform_(-bound, bound)  
#     #     model.model[2].weight.uniform_(-bound, bound)
#     #     model.model[2].bias.uniform_(-bound, bound)  
#     #     if len(model.model)>3: #ONLY SETUP FOR 2 hidden layers so far!
#     #         model.model[4].weight.uniform_(-bound, bound)
#     #         model.model[4].bias.uniform_(-bound, bound) 

#     criterion = nn.CrossEntropyLoss()
#     # optimizer = optim.SGD(model.parameters(), lr=0.25)
#     optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005
    
#     num_epochs = 4000
#     for epoch in range(num_epochs):
#         # Batch
#         outputs = model(X_tensor)
#         loss = criterion(outputs, y_tensor)
    
#         # #Stochastic
#         # outputs = model(torch.tensor(X[epoch%len(y)]).float())
#         # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

#         # Backward and optimize
#         optimizer.zero_grad()
#         loss.backward()

#         optimizer.step()
        
#     outputs_batch = model(X_tensor)
#     accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
#     print(f'Random Seed [{random_seed}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')
#     accuracies.append(accuracy)

In [ ]:
random_seed=11
torch.manual_seed(random_seed)  

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Initialize model and optimizer
model = BaarleNet([3,3])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005) #0.005


num_epochs = 12000
for epoch in range(num_epochs):
    # Batch
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)

    #Stochastic
    # outputs = model(torch.tensor(X[epoch%len(y)]).float())
    # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

    #Worth trying minibatch?
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # if epoch==307: break #Stop early here so weights don't get updated before we poke around
    
    optimizer.step()
    if (epoch+1) % (num_epochs//10) == 0:
        outputs_batch = model(X_tensor)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
fig=plt.figure(0,(8,8))
ax=fig.add_subplot(111)

probe=np.zeros((100,100,2))
for j, xx in enumerate(np.linspace(-1, 1, 100)):
    for k, yy in enumerate(np.linspace(-1, 1, 100)):
        probe[j, k]=[yy,xx]
probe=probe.reshape(100**2, -1)
probe_logits=model(torch.tensor(probe).float())
probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))

ax.imshow(map.mean(2), cmap='gray')
ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
           extent=[0, 960, 960, 0],  # This maps to image coordinates
           alpha=0.7,
           cmap='viridis')

In [ ]:
# torch.save(model.state_dict(), '../models/3_3_1.pth')

## 4x4

In [ ]:
accuracies=[]
for random_seed in range(32):
    torch.manual_seed(random_seed)
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.tensor(y)
    
    # Initialize model and optimizer
    model = BaarleNet([4,4])

    criterion = nn.CrossEntropyLoss()
    # optimizer = optim.SGD(model.parameters(), lr=0.25)
    optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005
    
    num_epochs = 4000
    for epoch in range(num_epochs):
        # Batch
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
    
        # #Stochastic
        # outputs = model(torch.tensor(X[epoch%len(y)]).float())
        # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        
    outputs_batch = model(X_tensor)
    accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
    print(f'Random Seed [{random_seed}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')
    accuracies.append(accuracy)

In [ ]:
random_seed=11
torch.manual_seed(random_seed)  

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Initialize model and optimizer
model = BaarleNet([4,4])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005) #0.005


num_epochs = 12000
for epoch in range(num_epochs):
    # Batch
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)

    #Stochastic
    # outputs = model(torch.tensor(X[epoch%len(y)]).float())
    # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

    #Worth trying minibatch?
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # if epoch==307: break #Stop early here so weights don't get updated before we poke around
    
    optimizer.step()
    if (epoch+1) % (num_epochs//10) == 0:
        outputs_batch = model(X_tensor)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
# fig=plt.figure(0,(8,8))
# ax=fig.add_subplot(111)

# probe=np.zeros((100,100,2))
# for j, xx in enumerate(np.linspace(-1, 1, 100)):
#     for k, yy in enumerate(np.linspace(-1, 1, 100)):
#         probe[j, k]=[yy,xx]
# probe=probe.reshape(100**2, -1)
# probe_logits=model(torch.tensor(probe).float())
# probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
# probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))

# ax.imshow(map.mean(2), cmap='gray')
# ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
#            extent=[0, 960, 960, 0],  # This maps to image coordinates
#            alpha=0.7,
#            cmap='viridis')

In [ ]:
# torch.save(model.state_dict(), '../models/4_4_1.pth')

In [ ]:
_=extract_weights_to_numpy(model)

## 2x2x2

In [ ]:
accuracies=[]
for random_seed in range(32):
    torch.manual_seed(random_seed)
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.tensor(y)
    
    # Initialize model and optimizer
    model = BaarleNet([2,2,2])
    
    #Larger weight init
    # bound=2.5
    # with torch.no_grad():
    #     model.model[0].weight.uniform_(-bound, bound)
    #     model.model[0].bias.uniform_(-bound, bound)  
    #     model.model[2].weight.uniform_(-bound, bound)
    #     model.model[2].bias.uniform_(-bound, bound)  
    #     if len(model.model)>3: #ONLY SETUP FOR 2 hidden layers so far!
    #         model.model[4].weight.uniform_(-bound, bound)
    #         model.model[4].bias.uniform_(-bound, bound) 

    criterion = nn.CrossEntropyLoss()
    # optimizer = optim.SGD(model.parameters(), lr=0.25)
    optimizer = optim.Adam(model.parameters(), lr=0.01) #0.005
    
    num_epochs = 4000
    for epoch in range(num_epochs):
        # Batch
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
    
        # #Stochastic
        # outputs = model(torch.tensor(X[epoch%len(y)]).float())
        # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        
    outputs_batch = model(X_tensor)
    accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
    print(f'Random Seed [{random_seed}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')
    accuracies.append(accuracy)

In [ ]:
random_seed=30
torch.manual_seed(random_seed)  

X_tensor = torch.FloatTensor(X)
y_tensor = torch.tensor(y)

# Initialize model and optimizer
model = BaarleNet([2,2,2])
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005) #0.005


num_epochs = 12000
for epoch in range(num_epochs):
    # Batch
    outputs = model(X_tensor)
    loss = criterion(outputs, y_tensor)

    #Stochastic
    # outputs = model(torch.tensor(X[epoch%len(y)]).float())
    # loss = criterion(outputs, torch.tensor(y[epoch%len(y)])) 

    #Worth trying minibatch?
    
    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()

    # if epoch==307: break #Stop early here so weights don't get updated before we poke around
    
    optimizer.step()
    if (epoch+1) % (num_epochs//10) == 0:
        outputs_batch = model(X_tensor)
        accuracy=(torch.argmax(outputs_batch, dim=1)==y_tensor).sum().item()/len(y)
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {accuracy:.4f}')


In [ ]:
fig=plt.figure(0,(8,8))
ax=fig.add_subplot(111)

probe=np.zeros((100,100,2))
for j, xx in enumerate(np.linspace(-1, 1, 100)):
    for k, yy in enumerate(np.linspace(-1, 1, 100)):
        probe[j, k]=[yy,xx]
probe=probe.reshape(100**2, -1)
probe_logits=model(torch.tensor(probe).float())
probe_logits=probe_logits.detach().numpy().reshape(100,100,2)
probe_softmax = torch.nn.Softmax(dim=1)(torch.tensor(probe_logits.reshape(-1, 2)))

ax.imshow(map.mean(2), cmap='gray')
ax.imshow(np.flipud(np.argmax(probe_logits,2)), 
           extent=[0, 960, 960, 0],  # This maps to image coordinates
           alpha=0.7,
           cmap='viridis')

In [ ]:
torch.save(model.state_dict(), '../models/2_2_2_1.pth')